In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
from pathlib import Path
import random
import json
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Device: {DEVICE}")

RAW_COLS = ["nli_raw_plot", "nli_raw_characters", "nli_raw_writing"]

def find_first(fname):
    hits = list(Path("/kaggle/input").rglob(fname))
    return hits[0] if hits else None

# Prefer enriched parquet
data_path = find_first("books_big_nli.parquet")

if data_path is not None:
    print(f"Using enriched file: {data_path}")
    df = pd.read_parquet(data_path)
else:
    base_path = find_first("books_big.parquet")
    item_nli_path = find_first("item_nli_aspects.parquet")

    if base_path is None:
        raise FileNotFoundError("Could not find books_big_nli.parquet or books_big.parquet in /kaggle/input")

    print(f"Using base file: {base_path}")
    df = pd.read_parquet(base_path)

    # If raw NLI cols are missing, try to merge item-level file
    if not all(c in df.columns for c in RAW_COLS):
        if item_nli_path is None:
            raise FileNotFoundError(
                "Raw NLI columns not found in books_big.parquet and item_nli_aspects.parquet is missing."
            )
        print(f"Merging item-level NLI features from: {item_nli_path}")
        item_nli = pd.read_parquet(item_nli_path)

        keep_cols = ["item_id"] + [c for c in RAW_COLS if c in item_nli.columns]
        missing_in_item = [c for c in RAW_COLS if c not in item_nli.columns]
        if missing_in_item:
            raise ValueError(f"item_nli_aspects.parquet is missing columns: {missing_in_item}")

        df = df.drop(columns=[c for c in RAW_COLS if c in df.columns], errors="ignore")
        df = df.merge(item_nli[keep_cols], on="item_id", how="left")

print(f"Rows: {len(df):,} | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
print("Columns:", df.columns.tolist())

required_cols = ["user_id", "item_id", "timestamp", "sentiment_score"] + RAW_COLS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# user/item mappings
user2idx = {uid: i for i, uid in enumerate(df["user_id"].unique())}
item2idx = {iid: i for i, iid in enumerate(df["item_id"].unique())}
idx2item = {v: k for k, v in item2idx.items()}

n_users = len(user2idx)
n_items = len(item2idx)

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

# temporal leave-one-out
df = df.sort_values(["user_id", "timestamp"]).copy()
df["_rank"] = df.groupby("user_id")["timestamp"].rank(method="first", ascending=False)

test_df  = df[df["_rank"] == 1].copy()
val_df   = df[df["_rank"] == 2].copy()
train_df = df[df["_rank"] > 2].copy()
df = df.drop(columns=["_rank"])

val_gt  = val_df.groupby("user_id")["item_id"].apply(set).to_dict()
test_gt = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

seen_items_val  = train_df.groupby("user_id")["item_idx"].apply(set).to_dict()
seen_items_test = pd.concat([train_df, val_df]).groupby("user_id")["item_idx"].apply(set).to_dict()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

In [ ]:
# item-level targets / feature tensors

# train only for all item-level targets
item_raw_df = (
    train_df.sort_values(["item_idx", "timestamp"])
            .groupby("item_idx")[RAW_COLS]
            .first()
            .reindex(range(n_items))
)

def build_feature_tensor(item_df, cols):
    values = item_df[cols].to_numpy(dtype=np.float32)
    mask = ~np.isnan(values)

    # z-score on observed values only
    for j in range(values.shape[1]):
        m = mask[:, j]
        if m.sum() > 0:
            mu = values[m, j].mean()
            sd = values[m, j].std() + 1e-8
            values[m, j] = (values[m, j] - mu) / sd
        values[~m, j] = 0.0

    feat_tensor = torch.tensor(values, dtype=torch.float32, device=DEVICE)
    mask_tensor = torch.tensor(mask, dtype=torch.bool, device=DEVICE)
    return feat_tensor, mask_tensor

# raw aspect targets / features
item_feat_raw, item_mask_raw = build_feature_tensor(item_raw_df, RAW_COLS)

# item-level sentiment target
item_sent_series = train_df.groupby("item_idx")["sentiment_score"].mean().reindex(range(n_items))
sent_values = item_sent_series.to_numpy(dtype=np.float32)
sent_mask = ~np.isnan(sent_values)

if sent_mask.sum() > 0:
    mu = sent_values[sent_mask].mean()
    sd = sent_values[sent_mask].std() + 1e-8
    sent_values[sent_mask] = (sent_values[sent_mask] - mu) / sd
sent_values[~sent_mask] = 0.0

item_sent_target = torch.tensor(sent_values, dtype=torch.float32, device=DEVICE)
item_sent_mask   = torch.tensor(sent_mask, dtype=torch.bool, device=DEVICE)

print("item_feat_raw:", item_feat_raw.shape)
print("item_mask_raw:", item_mask_raw.shape)
print("item_sent_target:", item_sent_target.shape)

print("\nRaw aspect coverage:")
for col in RAW_COLS:
    cov = item_raw_df[col].notna().mean() * 100
    print(f"  {col}: {cov:.1f}%")

print("\nSentiment coverage:", f"{item_sent_series.notna().mean() * 100:.1f}%")

In [ ]:
# metrics + dataset
def ndcg_at_k(recommended, relevant, k):
    if not relevant: return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(recommended, relevant, k):
    if not relevant: return 0.0
    return len(set(recommended[:k]) & relevant) / len(relevant)

def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & relevant) / k

def hr_at_k(recommended, relevant, k):
    return 1.0 if set(recommended[:k]) & relevant else 0.0

def evaluate_recommendations(recs, ground_truth, k=10):
    ndcgs, recalls, precisions, hrs = [], [], [], []
    for uid, rec_list in recs.items():
        gt = ground_truth.get(uid, set())
        if not gt: continue
        ndcgs.append(ndcg_at_k(rec_list, gt, k))
        recalls.append(recall_at_k(rec_list, gt, k))
        precisions.append(precision_at_k(rec_list, gt, k))
        hrs.append(hr_at_k(rec_list, gt, k))
    return {
        f"NDCG@{k}": float(np.mean(ndcgs)), f"Recall@{k}": float(np.mean(recalls)),
        f"Precision@{k}": float(np.mean(precisions)), f"HR@{k}": float(np.mean(hrs)),
        "n_users_evaluated": len(ndcgs),
    }

class BPRDataset(TorchDataset):
    def __init__(self, df, n_items):
        df = df[(df["user_idx"] >= 0) & (df["item_idx"] >= 0)].copy()
        self.users = df["user_idx"].values
        self.pos_items = df["item_idx"].values
        self.n_items = n_items
        self.user_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        user, pos = self.users[idx], self.pos_items[idx]
        seen = self.user_items.get(user, set())
        neg = random.randint(0, self.n_items - 1)
        while neg in seen: neg = random.randint(0, self.n_items - 1)
        return torch.tensor(user, dtype=torch.long), torch.tensor(pos, dtype=torch.long), torch.tensor(neg, dtype=torch.long)

In [ ]:
# aspect-mtl model
 
class AspectMTL(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        n_aspects=3,
        emb_dim_gmf=32,
        emb_dim_mlp=32,
        aspect_dim=16,
        mlp_layers=None,
        dropout=0.1,
    ):
        super().__init__()
        if mlp_layers is None:
            mlp_layers = [64, 32, 16]
 
        self.n_aspects = n_aspects
 
        self.gmf_user = nn.Embedding(n_users, emb_dim_gmf)
        self.gmf_item = nn.Embedding(n_items, emb_dim_gmf)
        self.mlp_user = nn.Embedding(n_users, emb_dim_mlp)
        self.mlp_item = nn.Embedding(n_items, emb_dim_mlp)

        self.user_query = nn.Embedding(n_users, aspect_dim)
        self.aspect_token_emb = nn.Embedding(n_aspects, aspect_dim)
        self.aspect_proj = nn.Linear(aspect_dim, emb_dim_mlp)
        self.rank_norm = nn.LayerNorm(emb_dim_mlp)
 
        # Ranking MLP
        mlp_input = emb_dim_mlp * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp = nn.Sequential(*layers)
 
        self.output_layer = nn.Linear(emb_dim_gmf + mlp_layers[-1], 1)

        self.aux_encoder = nn.Sequential(
            nn.Linear(emb_dim_mlp, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.sent_head = nn.Linear(32, 1)
        self.aspect_head = nn.Linear(32, n_aspects)
 
        self._init_weights()
 
    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item,
                     self.user_query, self.aspect_token_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
 
    def _aspect_attention(self, user_idx, aspect_vals, aspect_mask):
        token_ids = torch.arange(self.n_aspects, device=aspect_vals.device)
        token_base = self.aspect_token_emb(token_ids).unsqueeze(0).expand(aspect_vals.shape[0], -1, -1)
 
        token_repr = aspect_vals.unsqueeze(-1) * token_base
        q = self.user_query(user_idx).unsqueeze(1)
 
        scores = (token_repr * q).sum(dim=-1) / np.sqrt(token_repr.shape[-1])
        scores = scores.masked_fill(~aspect_mask, -1e9)
 
        attn = torch.softmax(scores, dim=-1)
        valid_any = aspect_mask.any(dim=1, keepdim=True)
        attn = torch.where(valid_any, attn, torch.zeros_like(attn))
 
        out = (attn.unsqueeze(-1) * token_repr).sum(dim=1)
        return self.aspect_proj(out)
 
    def forward_rank(self, user_idx, item_idx, item_feat_raw, item_mask_raw):
        gmf_out = self.gmf_user(user_idx) * self.gmf_item(item_idx)
 
        item_asp = self._aspect_attention(
            user_idx, item_feat_raw[item_idx], item_mask_raw[item_idx]
        )
        item_asp = self.rank_norm(item_asp)
        item_fused = self.mlp_item(item_idx) + item_asp
 
        mlp_in = torch.cat([self.mlp_user(user_idx), item_fused], dim=-1)
        mlp_out = self.mlp(mlp_in)
 
        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.output_layer(combined).squeeze(-1)
 
    def forward_aux(self, item_idx):
        item_latent = self.mlp_item(item_idx)
        h = self.aux_encoder(item_latent)
        sent_pred = self.sent_head(h).squeeze(-1)
        asp_pred = self.aspect_head(h)
        return sent_pred, asp_pred

In [ ]:
# config

EMB_DIM_GMF = 32
EMB_DIM_MLP = 32
ASPECT_DIM = 16
MLP_LAYERS = [64, 32, 16]
DROPOUT = 0.1
 
LR = 1e-3
WEIGHT_DECAY = 1e-5
N_EPOCHS = 50
BATCH_SIZE = 2048
PATIENCE = 4
EVAL_EVERY = 2
K = 10
 
LAMBDA_SENT = 0.01
LAMBDA_ASP = 0.02
 
dataset = BPRDataset(train_df, n_items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
all_item_idxs = np.arange(n_items)
 
VAL_USERS_FIXED = list(val_gt.keys())
if len(VAL_USERS_FIXED) > 3000:
    random.seed(SEED)
    VAL_USERS_FIXED = random.sample(VAL_USERS_FIXED, 3000)
 
def masked_mse(pred, target, mask):
    if mask.sum() == 0:
        return pred.new_tensor(0.0)
    return ((pred[mask] - target[mask]) ** 2).mean()
 
def train_and_evaluate(model, model_name):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_ndcg = -1.0
    no_improve = 0
    best_state = None
    history = []
 
    print(f"\nTraining {model_name}...")
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Lambdas: sent={LAMBDA_SENT}, asp={LAMBDA_ASP}\n")
 
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_bpr = 0.0
        total_sent = 0.0
        total_asp = 0.0
 
        for users_b, pos_b, neg_b in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            users_b = users_b.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            neg_b = neg_b.to(DEVICE)
 
            # Ranking loss
            pos_score = model.forward_rank(users_b, pos_b, item_feat_raw, item_mask_raw)
            neg_score = model.forward_rank(users_b, neg_b, item_feat_raw, item_mask_raw)
            bpr_loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
 
            # Aux losses on unique positive items only
            pos_unique = pos_b.unique()
            sent_pred, asp_pred = model.forward_aux(pos_unique)
 
            sent_loss = masked_mse(sent_pred, item_sent_target[pos_unique], item_sent_mask[pos_unique])
            asp_loss = masked_mse(asp_pred, item_feat_raw[pos_unique], item_mask_raw[pos_unique])
 
            loss = bpr_loss + LAMBDA_SENT * sent_loss + LAMBDA_ASP * asp_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            bs = len(users_b)
            total_loss += loss.item() * bs
            total_bpr += bpr_loss.item() * bs
            total_sent += sent_loss.item() * bs
            total_asp += asp_loss.item() * bs
 
        n_total = len(dataset)
        avg_loss = total_loss / n_total
        avg_bpr = total_bpr / n_total
        avg_sent = total_sent / n_total
        avg_asp = total_asp / n_total
 
        if epoch % EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            scores_list = []
            with torch.no_grad():
                all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
                for uid in VAL_USERS_FIXED:
                    uidx = user2idx.get(uid, -1)
                    if uidx < 0:
                        continue
                    user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
                    sc = model.forward_rank(user_t, all_items_t, item_feat_raw, item_mask_raw).cpu().numpy()
                    for sidx in seen_items_val.get(uid, set()):
                        sc[sidx] = -1e9
                    top = np.argsort(sc)[::-1][:K]
                    rec = [idx2item[i] for i in top]
                    scores_list.append(ndcg_at_k(rec, val_gt[uid], K))
 
            val_ndcg = float(np.mean(scores_list)) if scores_list else 0.0
            print(
                f"  Epoch {epoch:>3}/{N_EPOCHS}  "
                f"Loss: {avg_loss:.4f}  BPR: {avg_bpr:.4f}  "
                f"Sent: {avg_sent:.4f}  Asp: {avg_asp:.4f}  "
                f"val_NDCG@{K}: {val_ndcg:.4f}"
            )
            history.append({
                "epoch": epoch, "loss": avg_loss, "bpr_loss": avg_bpr,
                "sent_loss": avg_sent, "asp_loss": avg_asp, "val_ndcg": val_ndcg,
            })
 
            if val_ndcg > best_ndcg + 1e-5:
                best_ndcg = val_ndcg
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"\n  Early stopping at epoch {epoch} (best: {best_ndcg:.4f})")
                    break
 
    if best_state:
        model.load_state_dict(best_state)
        model.to(DEVICE)
 
    print(f"  Best val NDCG@{K}: {best_ndcg:.4f}")
 
    @torch.no_grad()
    def generate_recs(target_users, seen_dict):
        model.eval()
        all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
        recs = {}
        for uid in tqdm(target_users, desc="Recs", leave=False):
            uidx = user2idx.get(uid, -1)
            if uidx < 0:
                continue
            user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
            sc = model.forward_rank(user_t, all_items_t, item_feat_raw, item_mask_raw).cpu().numpy()
            for sidx in seen_dict.get(uid, set()):
                sc[sidx] = -1e9
            top = np.argsort(sc)[::-1][:K]
            recs[uid] = [idx2item[i] for i in top]
        return recs
 
    print(f"\nEvaluating {model_name} on VAL...")
    val_recs = generate_recs(val_df["user_id"].unique().tolist(), seen_items_val)
    val_metrics = evaluate_recommendations(val_recs, val_gt, k=K)
 
    print(f"Evaluating {model_name} on TEST...")
    test_recs = generate_recs(test_df["user_id"].unique().tolist(), seen_items_test)
    test_metrics = evaluate_recommendations(test_recs, test_gt, k=K)
 
    print(f"\n{'='*40}")
    print(f"{model_name} — VAL")
    print(f"{'='*40}")
    for key, val in val_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")
 
    print(f"\n{model_name} — TEST")
    print(f"{'='*40}")
    for key, val in test_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")
 
    fname = model_name.lower().replace("+", "_").replace("-", "_").replace(" ", "_").replace("(", "").replace(")", "")
    results = {"model": model_name, "val": val_metrics, "test": test_metrics}
    with open(OUT_DIR / f"{fname}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    torch.save(model.state_dict(), OUT_DIR / f"{fname}_model.pt")
    pd.DataFrame(history).to_csv(OUT_DIR / f"{fname}_history.csv", index=False)
    print(f"Saved: {fname}")
 
    return val_metrics, test_metrics, history

In [ ]:
# warm start mtl expetiments
 
# best Aspect-CF raw weights
aspect_cf_path = None
for p in Path("/kaggle/input").rglob("aspect_cf_*raw*_model.pt"):
    aspect_cf_path = p
    break
if aspect_cf_path is None:
    for p in Path("/kaggle/working").rglob("aspect_cf_*raw*_model.pt"):
        aspect_cf_path = p
        break
 
if aspect_cf_path is None:
    print("WARNING: Aspect-CF raw model not found! Training from scratch.")
    WARM_START = False
else:
    print(f"Found Aspect-CF raw weights: {aspect_cf_path}")
    aspect_cf_state = torch.load(aspect_cf_path, map_location="cpu")
    WARM_START = True
 
def make_warmstarted_mtl():
    m = AspectMTL(
        n_users=n_users,
        n_items=n_items,
        n_aspects=3,
        emb_dim_gmf=EMB_DIM_GMF,
        emb_dim_mlp=EMB_DIM_MLP,
        aspect_dim=ASPECT_DIM,
        mlp_layers=MLP_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)
 
    if WARM_START:
        mtl_state = m.state_dict()
        loaded = 0
        skipped = 0
        for key, val in aspect_cf_state.items():
            if key in mtl_state and mtl_state[key].shape == val.shape:
                mtl_state[key] = val
                loaded += 1
            else:
                skipped += 1
        m.load_state_dict(mtl_state)
        print(f"  Warm-start: loaded {loaded} params, skipped {skipped} (aux heads)")
 
    return m
 
experiments = [
    {"name": "MTL-sent-only", "lambda_sent": 0.005, "lambda_asp": 0.0},
    {"name": "MTL-asp-only",  "lambda_sent": 0.0,   "lambda_asp": 0.01},
    {"name": "MTL-both-soft", "lambda_sent": 0.005, "lambda_asp": 0.01},
]
 
WS_LR = 3e-4
WS_EPOCHS = 20
WS_PATIENCE = 4
WS_EVAL_EVERY = 2
 
all_results = []
 
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"{exp['name']}  (sent={exp['lambda_sent']}, asp={exp['lambda_asp']})")
    print(f"{'='*60}")
 
    model = make_warmstarted_mtl()
    optimizer = torch.optim.Adam(model.parameters(), lr=WS_LR, weight_decay=WEIGHT_DECAY)
 
    best_ndcg = -1.0
    no_improve = 0
    best_state = None
    history = []
 
    lam_s = exp["lambda_sent"]
    lam_a = exp["lambda_asp"]
 
    for epoch in range(1, WS_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_bpr = 0.0
        total_sent = 0.0
        total_asp = 0.0
 
        for users_b, pos_b, neg_b in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            users_b = users_b.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            neg_b = neg_b.to(DEVICE)
 
            pos_score = model.forward_rank(users_b, pos_b, item_feat_raw, item_mask_raw)
            neg_score = model.forward_rank(users_b, neg_b, item_feat_raw, item_mask_raw)
            bpr_loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
 
            sent_loss = torch.tensor(0.0, device=DEVICE)
            asp_loss = torch.tensor(0.0, device=DEVICE)
 
            if lam_s > 0 or lam_a > 0:
                pos_unique = pos_b.unique()
                sent_pred, asp_pred = model.forward_aux(pos_unique)
 
                if lam_s > 0:
                    sent_loss = masked_mse(sent_pred, item_sent_target[pos_unique], item_sent_mask[pos_unique])
                if lam_a > 0:
                    asp_loss = masked_mse(asp_pred, item_feat_raw[pos_unique], item_mask_raw[pos_unique])
 
            loss = bpr_loss + lam_s * sent_loss + lam_a * asp_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            bs = len(users_b)
            total_loss += loss.item() * bs
            total_bpr += bpr_loss.item() * bs
            total_sent += sent_loss.item() * bs
            total_asp += asp_loss.item() * bs
 
        n_total = len(dataset)
 
        if epoch % WS_EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            scores_list = []
            with torch.no_grad():
                all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
                for uid in VAL_USERS_FIXED:
                    uidx = user2idx.get(uid, -1)
                    if uidx < 0:
                        continue
                    user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
                    sc = model.forward_rank(user_t, all_items_t, item_feat_raw, item_mask_raw).cpu().numpy()
                    for sidx in seen_items_val.get(uid, set()):
                        sc[sidx] = -1e9
                    top = np.argsort(sc)[::-1][:K]
                    rec = [idx2item[i] for i in top]
                    scores_list.append(ndcg_at_k(rec, val_gt[uid], K))
 
            val_ndcg = float(np.mean(scores_list)) if scores_list else 0.0
            print(
                f"  Epoch {epoch:>2}/{WS_EPOCHS}  "
                f"BPR: {total_bpr/n_total:.4f}  "
                f"Sent: {total_sent/n_total:.4f}  Asp: {total_asp/n_total:.4f}  "
                f"NDCG: {val_ndcg:.4f}"
            )
            history.append({"epoch": epoch, "val_ndcg": val_ndcg})
 
            if val_ndcg > best_ndcg + 1e-5:
                best_ndcg = val_ndcg
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= WS_PATIENCE:
                    print(f"  Early stop (best: {best_ndcg:.4f})")
                    break
 
    if best_state:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    @torch.no_grad()
    def generate_recs(target_users, seen_dict):
        model.eval()
        all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
        recs = {}
        for uid in tqdm(target_users, desc="Recs", leave=False):
            uidx = user2idx.get(uid, -1)
            if uidx < 0:
                continue
            user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
            sc = model.forward_rank(user_t, all_items_t, item_feat_raw, item_mask_raw).cpu().numpy()
            for sidx in seen_dict.get(uid, set()):
                sc[sidx] = -1e9
            top = np.argsort(sc)[::-1][:K]
            recs[uid] = [idx2item[i] for i in top]
        return recs
 
    val_recs = generate_recs(val_df["user_id"].unique().tolist(), seen_items_val)
    val_metrics = evaluate_recommendations(val_recs, val_gt, k=K)
 
    test_recs = generate_recs(test_df["user_id"].unique().tolist(), seen_items_test)
    test_metrics = evaluate_recommendations(test_recs, test_gt, k=K)
 
    print(f"\n  {exp['name']} — VAL  NDCG@10={val_metrics['NDCG@10']:.4f}  HR@10={val_metrics['HR@10']:.4f}")
    print(f"  {exp['name']} — TEST NDCG@10={test_metrics['NDCG@10']:.4f}  HR@10={test_metrics['HR@10']:.4f}")
 
    all_results.append({
        "name": exp["name"],
        "val_ndcg": val_metrics["NDCG@10"],
        "val_hr": val_metrics["HR@10"],
        "test_ndcg": test_metrics["NDCG@10"],
        "test_hr": test_metrics["HR@10"],
        "quick_best": best_ndcg,
    })
 
    fname = exp["name"].lower().replace("-", "_")
    torch.save(model.state_dict(), OUT_DIR / f"{fname}_model.pt")
    results = {"model": exp["name"], "val": val_metrics, "test": test_metrics}
    with open(OUT_DIR / f"{fname}_results.json", "w") as f:
        json.dump(results, f, indent=2)
 
print(f"\n{'='*70}")
print("WARM-START MTL ABLATION SUMMARY")
print(f"{'='*70}")
print(f"{'Model':<25} {'Quick Val':>10} {'Full Val':>10} {'Test':>10}")
print(f"{'-'*55}")
print(f"{'Aspect-CF raw (ref)':<25} {'—':>10} {'0.0476':>10} {'0.0301':>10}")
for r in all_results:
    print(f"{r['name']:<25} {r['quick_best']:>10.4f} {r['val_ndcg']:>10.4f} {r['test_ndcg']:>10.4f}")
print(f"{'='*70}")
 
pd.DataFrame(all_results).to_csv(OUT_DIR / "mtl_ablation_summary.csv", index=False)
print("Saved: mtl_ablation_summary.csv")